Space time diagram generation(E=W and N=S KEEPING ALL OTHER RMTS 0)

In [ ]:
# ============================================================
# CODE 1
# GENERATE + SAVE FULL 300 x 300 SNAPSHOTS
#
# CASE:
#       E = W
#       N = S
#
# FULL UNIVERSE:
#       3000 x 3000
#
# ANALYTICAL SNAPSHOT:
#       CENTRAL 300 x 300
#
# RULES:
#       256
#
# TIME:
#       100
#
# SAVES:
#       Rule_000.npy ... Rule_255.npy
#
# EACH FILE:
#       shape = (100, 300, 300)
#
# NO MIDDLE ROW
# ============================================================

import numpy as np
import os
import gc
from tqdm import tqdm


# ============================================================
# PARAMETERS
# ============================================================

N_ROWS = 3000
N_COLS = 3000

CROP_ROWS = 300
CROP_COLS = 300

TIME_STEPS = 100
NUM_RULES = 256

OUTPUT_FOLDER = "CA_Snapshots_256"

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)


# ============================================================
# CENTRAL 300 x 300 CROP
# ============================================================

r0 = (
    N_ROWS - CROP_ROWS
) // 2

r1 = r0 + CROP_ROWS

c0 = (
    N_COLS - CROP_COLS
) // 2

c1 = c0 + CROP_COLS


# ============================================================
# SAME INITIAL GRID FOR ALL RULES
# ============================================================

rng = np.random.default_rng(42)

initial_grid = rng.integers(
    0,
    2,
    size=(N_ROWS, N_COLS),
    dtype=np.uint8
)

print(
    "Initial grid:",
    initial_grid.shape
)


# ============================================================
# VALID RMTs
#
# ORDER:
#       N W C E S
#
# CONDITION:
#       N = S
#       E = W
# ============================================================

valid_rmts = []

for N in [0, 1]:

    for W in [0, 1]:

        for C in [0, 1]:

            for E in [0, 1]:

                for S in [0, 1]:

                    if N == S and E == W:

                        valid_rmts.append(
                            (N, W, C, E, S)
                        )


assert len(valid_rmts) == 8


# ============================================================
# GENERATE 256 RULES
#
# 8 VALID RMTs
# 24 OTHER RMTs = 0
# ============================================================

rmt_rules = []

for rule_number in range(
    NUM_RULES
):

    logical_rule = format(
        rule_number,
        "08b"
    )

    rule = np.zeros(
        32,
        dtype=np.uint8
    )

    for rmt, output in zip(
        valid_rmts,
        logical_rule
    ):

        N, W, C, E, S = rmt

        index = (
            (N << 4)
            |
            (W << 3)
            |
            (C << 2)
            |
            (E << 1)
            |
            S
        )

        rule[index] = int(output)

    rmt_rules.append(rule)


rmt_rules = np.array(
    rmt_rules,
    dtype=np.uint8
)


# ============================================================
# EVOLUTION
# ============================================================

def evolve(
    grid,
    rule
):

    N = np.roll(
        grid,
        1,
        axis=0
    )

    S = np.roll(
        grid,
        -1,
        axis=0
    )

    W = np.roll(
        grid,
        1,
        axis=1
    )

    E = np.roll(
        grid,
        -1,
        axis=1
    )

    index = (
        (N << 4)
        |
        (W << 3)
        |
        (grid << 2)
        |
        (E << 1)
        |
        S
    )

    new_grid = rule[index]

    del N, S, W, E, index

    return new_grid


# ============================================================
# PROCESS EACH RULE
# ============================================================

for rule_number in tqdm(
    range(NUM_RULES),
    desc="Generating rules"
):

    output_file = os.path.join(
        OUTPUT_FOLDER,
        f"Rule_{rule_number:03d}.npy"
    )


    # --------------------------------------------------------
    # RESUME SUPPORT
    # --------------------------------------------------------

    if os.path.exists(
        output_file
    ):

        print(
            f"Skipping Rule {rule_number} "
            "(already saved)"
        )

        continue


    grid = initial_grid.copy()


    # --------------------------------------------------------
    # STORE ONLY THE 300 x 300 SNAPSHOTS
    #
    # NOT the 3000 x 3000 grids
    # --------------------------------------------------------

    snapshots = np.empty(
        (
            TIME_STEPS,
            CROP_ROWS,
            CROP_COLS
        ),
        dtype=np.uint8
    )


    # ========================================================
    # 100 TIME STEPS
    # ========================================================

    for t in range(
        TIME_STEPS
    ):

        # FULL 300 x 300 SNAPSHOT
        snapshots[t] = grid[
            r0:r1,
            c0:c1
        ]


        # EVOLVE FULL 3000 x 3000 UNIVERSE
        grid = evolve(
            grid,
            rmt_rules[
                rule_number
            ]
        )


    # --------------------------------------------------------
    # SAVE:
    #
    # (100, 300, 300)
    # --------------------------------------------------------

    np.save(
        output_file,
        snapshots
    )


    print(
        f"Rule {rule_number} saved:",
        snapshots.shape
    )


    # --------------------------------------------------------
    # RELEASE MEMORY
    # --------------------------------------------------------

    del snapshots
    del grid

    gc.collect()


print("\n======================================")
print("CODE 1 COMPLETE")
print("======================================")
print(
    "Saved folder:",
    OUTPUT_FOLDER
)
print(
    "Each rule:",
    "(100, 300, 300)"
)
print(
    "Total rules:",
    NUM_RULES
)

Initial grid: (3000, 3000)


Generating rules:   0%|          | 1/256 [00:09<42:13,  9.93s/it]

Rule 0 saved: (100, 300, 300)


Generating rules:   1%|          | 2/256 [00:15<31:31,  7.45s/it]

Rule 1 saved: (100, 300, 300)


...

Rule 253 saved: (100, 300, 300)


Generating rules: 100%|█████████▉| 255/256 [23:28<00:05,  5.62s/it]

Rule 254 saved: (100, 300, 300)


Generating rules: 100%|██████████| 256/256 [23:34<00:00,  5.53s/it]

Rule 255 saved: (100, 300, 300)

CODE 1 COMPLETE
Saved folder: CA_Snapshots_256
Each rule: (100, 300, 300)
Total rules: 256


In [ ]:
# ============================================================
# COMBINE ALL GENERATED RULE PDFs INTO ONE PDF
# ============================================================

import os
from pypdf import PdfWriter

PDF_FOLDER = "Rule_PDFs"
OUTPUT_FILE = "Rules_000_to_195_Combined.pdf"

writer = PdfWriter()

# Rule 000 to Rule 195
for rule in range(196):

    file_path = os.path.join(
        PDF_FOLDER,
        f"Rule_{rule:03d}.pdf"
    )

    if os.path.exists(file_path):

        print(f"Adding Rule {rule}")

        writer.append(file_path)

    else:

        print(f"Missing Rule {rule}")

# Save combined PDF
with open(OUTPUT_FILE, "wb") as f:
    writer.write(f)

writer.close()

print("\n===================================")
print("COMBINATION COMPLETE")
print("===================================")
print(f"Output: {OUTPUT_FILE}")
print("Pages should be: 196")

Adding Rule 0
Adding Rule 1
Adding Rule 2
Adding Rule 3
Adding Rule 4
Adding Rule 5
Adding Rule 6
Adding Rule 7
Adding Rule 8
Adding Rule 9
Adding Rule 10
Adding Rule 11
Adding Rule 12
Adding Rule 13
Adding Rule 14
Adding Rule 15
Adding Rule 16
Adding Rule 17
Adding Rule 18
Adding Rule 19
Adding Rule 20
Adding Rule 21
Adding Rule 22
Adding Rule 23
Adding Rule 24
Adding Rule 25
Adding Rule 26
Adding Rule 27
Adding Rule 28
Adding Rule 29
Adding Rule 30
Adding Rule 31
Adding Rule 32
Adding Rule 33
Adding Rule 34
Adding Rule 35
Adding Rule 36
Adding Rule 37
Adding Rule 38
Adding Rule 39
Adding Rule 40
Adding Rule 41
Adding Rule 42
Adding Rule 43
Adding Rule 44
Adding Rule 45
Adding Rule 46
Adding Rule 47
Adding Rule 48
Adding Rule 49
Adding Rule 50
Adding Rule 51
Adding Rule 52
Adding Rule 53
Adding Rule 54
Adding Rule 55
Adding Rule 56
Adding Rule 57
Adding Rule 58
Adding Rule 59
Adding Rule 60
Adding Rule 61
Adding Rule 62
Adding Rule 63
Adding Rule 64
Adding Rule 65
Adding Rule 66
Addin

Type of rule(index-000-195)

In [ ]:
# ============================================================
# CA NATURE CLASSIFICATION
#
# CLASSIFICATION:
#       1. Fixed Point
#       2. Periodic
#       3. Unknown
#
# INPUT:
#       Saved .npy CA snapshots
#
# CURRENTLY GENERATED:
#       Rule 000 to Rule 195
#       = 196 rules
#
# EACH FILE:
#       (100, 300, 300)
#
# IMPORTANT:
#       FULL 300 x 300 MATRIX IS USED
#       NO MIDDLE ROW
#       NO CA REGENERATION
# ============================================================


import numpy as np
import pandas as pd
import os
from tqdm import tqdm


# ============================================================
# 1. PARAMETERS
# ============================================================

INPUT_FOLDER = "CA_Snapshots_256"

FIRST_RULE = 0
LAST_RULE = 195

TIME_STEPS = 100

GRID_ROWS = 300
GRID_COLS = 300


# ============================================================
# 2. CHECK TWO COMPLETE SNAPSHOTS
# ============================================================

def same_snapshot(
    snapshot1,
    snapshot2
):

    return np.array_equal(
        snapshot1,
        snapshot2
    )


# ============================================================
# 3. DETECT FIXED POINT / PERIODIC / UNKNOWN
#
# First-recurrence method:
#
# CA evolution is deterministic, so if S(t) = S(t+k) for any
# t, k, the trajectory is provably periodic with period k
# from t onward. No repeat-counting is needed.
#
# cycle == 1  -> Fixed Point
# cycle >= 2  -> Periodic
# never found -> Unknown
# ============================================================

def detect_nature(
    snapshots
):

    n = len(
        snapshots
    )

    state_history = {}

    for step in range(n):

        current_state = snapshots[step].tobytes()

        if current_state in state_history:

            transient = state_history[current_state]
            cycle = step - transient

            if cycle == 1:

                return "Fixed Point", transient, cycle

            else:

                return "Periodic", transient, cycle

        else:

            state_history[current_state] = step

    return "Unknown", -1, 0


# ============================================================
# 5. PROCESS RULES
# ============================================================

results = []


print("\n==========================================")
print("CA NATURE ANALYSIS")
print("==========================================")

print(
    f"Rules analysed: "
    f"{FIRST_RULE} to {LAST_RULE}"
)

print(
    f"Total rules: "
    f"{LAST_RULE - FIRST_RULE + 1}"
)

print(
    f"Grid used: "
    f"{GRID_ROWS} x {GRID_COLS}"
)

print(
    f"Time steps: "
    f"{TIME_STEPS}"
)


# ============================================================
# LOOP THROUGH RULES
# ============================================================

for rule_number in tqdm(
    range(
        FIRST_RULE,
        LAST_RULE + 1
    ),
    desc="Analysing rules"
):

    # --------------------------------------------------------
    # File name
    # --------------------------------------------------------

    file_path = os.path.join(
        INPUT_FOLDER,
        f"Rule_{rule_number:03d}.npy"
    )


    # --------------------------------------------------------
    # Check file
    # --------------------------------------------------------

    if not os.path.exists(
        file_path
    ):

        print(
            f"\nWARNING: "
            f"Rule {rule_number} file not found."
        )

        continue


    # ========================================================
    # LOAD SAVED SNAPSHOTS
    # ========================================================

    snapshots = np.load(
        file_path
    )


    # ========================================================
    # CHECK SHAPE
    # ========================================================

    if snapshots.shape != (
        TIME_STEPS,
        GRID_ROWS,
        GRID_COLS
    ):

        print(
            f"\nWARNING: Rule {rule_number} "
            f"has shape {snapshots.shape}"
        )

        del snapshots

        continue


    # ========================================================
    # NATURE DETECTION (Fixed Point / Periodic / Unknown)
    # ========================================================

    (
        nature,
        transient_length,
        cycle_length
    ) = detect_nature(
        snapshots
    )


    # ========================================================
    # EXTRA INFORMATION
    # ========================================================

    # --------------------------------------------------------
    # Final density
    # --------------------------------------------------------

    final_density = float(
        snapshots[-1].mean()
    )


    # --------------------------------------------------------
    # Mean density
    # --------------------------------------------------------

    mean_density = float(
        snapshots.mean()
    )


    # --------------------------------------------------------
    # State change rates
    #
    # Complete 300 x 300 matrices are compared.
    # --------------------------------------------------------

    change_rates = []

    for t in range(
        1,
        TIME_STEPS
    ):

        change_rate = np.mean(
            snapshots[t]
            !=
            snapshots[t - 1]
        )

        change_rates.append(
            change_rate
        )


    change_rates = np.array(
        change_rates
    )


    mean_state_change_rate = float(
        change_rates.mean()
    )


    final_state_change_rate = float(
        change_rates[-1]
    )


    # ========================================================
    # STORE RESULT
    # ========================================================

    results.append({

        "Rule":
            rule_number,

        "Nature":
            nature,

        "Transient_Length":
            transient_length,

        "Cycle_Length":
            cycle_length,

        "Final_Density":
            final_density,

        "Mean_Density":
            mean_density,

        "Mean_State_Change_Rate":
            mean_state_change_rate,

        "Final_State_Change_Rate":
            final_state_change_rate

    })


    # ========================================================
    # FREE MEMORY
    # ========================================================

    del snapshots


# ============================================================
# 6. CREATE DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)


# ============================================================
# 7. SORT BY RULE NUMBER
# ============================================================

results_df = results_df.sort_values(
    "Rule"
).reset_index(
    drop=True
)


# ============================================================
# 8. SAVE CSV
# ============================================================

CSV_FILE = (
    "CA_Nature_Analysis_Rules_000_to_195.csv"
)

results_df.to_csv(
    CSV_FILE,
    index=False
)


# ============================================================
# 9. SAVE EXCEL
# ============================================================

EXCEL_FILE = (
    "CA_Nature_Analysis_Rules_000_to_195.xlsx"
)

results_df.to_excel(
    EXCEL_FILE,
    index=False
)


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print("\n==========================================")
print("ANALYSIS COMPLETE")
print("==========================================")

print(
    results_df[
        [
            "Rule",
            "Nature",
            "Transient_Length",
            "Cycle_Length"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 11. NATURE COUNTS
# ============================================================

print("\n==========================================")
print("NATURE COUNTS")
print("==========================================")

nature_counts = (
    results_df[
        "Nature"
    ].value_counts()
)


print(
    nature_counts
)


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

print("\n==========================================")
print("FINAL SUMMARY")
print("==========================================")

print(
    "Rules analysed:",
    len(results_df)
)

print(
    "Fixed Point:",
    int(
        (
            results_df["Nature"]
            == "Fixed Point"
        ).sum()
    )
)

print(
    "Periodic:",
    int(
        (
            results_df["Nature"]
            == "Periodic"
        ).sum()
    )
)

print(
    "Unknown:",
    int(
        (
            results_df["Nature"]
            == "Unknown"
        ).sum()
    )
)

print("\nFiles saved:")

print(
    CSV_FILE
)

print(
    EXCEL_FILE
)


CA NATURE ANALYSIS
Rules analysed: 0 to 195
Total rules: 196
Grid used: 300 x 300
Time steps: 100


Analysing rules: 100%|██████████| 196/196 [00:21<00:00,  8.97it/s]



ANALYSIS COMPLETE
 Rule      Nature  Transient_Length  Cycle_Length
    0 Fixed Point                 1             1
    1 Fixed Point                 3             1
    2 Fixed Point                 3             1
    3 Fixed Point                 3             1
    4 Fixed Point                 4             1
    5 Fixed Point                 5             1
    6 Fixed Point                 4             1
    7 Fixed Point                 5             1
    8 Fixed Point                 5             1
    9 Fixed Point                 5             1
   10 Fixed Point                 5             1
   11 Fixed Point                 6             1
   12 Fixed Point                 5             1
   13 Fixed Point                 7             1
   14 Fixed Point                 8             1
   15 Fixed Point                 8             1
   16 Fixed Point                 4             1
   17 Fixed Point                 5             1
   18 Fixed Point              

Full space time diagram pdf for any rule(000-195)

In [ ]:
# ============================================================
# 100-PAGE SPACE-TIME VISUALIZATION FOR ONE RULE
#
# CHANGE ONLY:
#       RULE_NO = 193
#
# Page 1   -> t = 0
# Page 2   -> t = 1
# ...
# Page 100 -> t = 99
#
# EACH PAGE:
#       COMPLETE 300 x 300 GRID
#
# NO MIDDLE ROW
# NO CA REGENERATION
# ============================================================

import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


# ============================================================
# CHANGE ONLY THIS NUMBER
# ============================================================

RULE_NO = 206


# ============================================================
# SETTINGS
# ============================================================

INPUT_FOLDER = "CA_Snapshots_256"

START_TIME = 0
END_TIME = 99


# ============================================================
# CREATE FILE NAME
# ============================================================

input_file = os.path.join(
    INPUT_FOLDER,
    f"Rule_{RULE_NO:03d}.npy"
)

output_file = (
    f"Rule_{RULE_NO:03d}_100_Steps.pdf"
)


# ============================================================
# CHECK FILE
# ============================================================

if not os.path.exists(input_file):

    raise FileNotFoundError(
        f"Rule file not found:\n{input_file}"
    )


# ============================================================
# LOAD ONE RULE
# ============================================================

snapshots = np.load(
    input_file
)


# ============================================================
# CHECK DATA
# ============================================================

print(
    "Loaded:",
    input_file
)

print(
    "Shape:",
    snapshots.shape
)


if snapshots.shape != (
    100,
    300,
    300
):

    raise ValueError(
        f"Expected shape (100, 300, 300), "
        f"but got {snapshots.shape}"
    )


# ============================================================
# CREATE 100-PAGE PDF
# ============================================================

with PdfPages(output_file) as pdf:

    for t in range(
        START_TIME,
        END_TIME + 1
    ):

        # ----------------------------------------------------
        # COMPLETE 300 x 300 MATRIX
        # ----------------------------------------------------

        grid = snapshots[t]


        # ----------------------------------------------------
        # CREATE PAGE
        # ----------------------------------------------------

        fig, ax = plt.subplots(
            figsize=(8, 8)
        )


        # ----------------------------------------------------
        # DISPLAY FULL 300 x 300 GRID
        # ----------------------------------------------------

        ax.imshow(
            grid,
            cmap="binary",
            interpolation="nearest",
            aspect="equal"
        )


        # ----------------------------------------------------
        # TITLE
        # ----------------------------------------------------

        ax.set_title(
            f"Rule {RULE_NO}    |    "
            f"Time Step t = {t}\n"
            f"Full 300 × 300 Configuration",
            fontsize=14
        )


        # ----------------------------------------------------
        # AXES
        # ----------------------------------------------------

        ax.set_xlabel(
            "Spatial Column (0–299)"
        )

        ax.set_ylabel(
            "Spatial Row (0–299)"
        )


        # ----------------------------------------------------
        # SAVE PAGE
        # ----------------------------------------------------

        plt.tight_layout()

        pdf.savefig(
            fig,
            dpi=120
        )

        plt.close(fig)


# ============================================================
# FREE MEMORY
# ============================================================

del snapshots


# ============================================================
# DONE
# ============================================================

print("\n==========================================")
print("PDF CREATED SUCCESSFULLY")
print("==========================================")

print(
    "Rule:",
    RULE_NO
)

print(
    "Time steps:",
    "0 to 99"
)

print(
    "Pages:",
    100
)

print(
    "Grid per page:",
    "300 x 300"
)

print(
    "Output:",
    output_file
)

Loaded: CA_Snapshots_256/Rule_206.npy
Shape: (100, 300, 300)

PDF CREATED SUCCESSFULLY
Rule: 206
Time steps: 0 to 99
Pages: 100
Grid per page: 300 x 300
Output: Rule_206_100_Steps.pdf
